In [20]:
# !pip -q install transformers accelerate sentencepiece spacy nltk
# !python -m spacy download en_core_web_sm -q


In [21]:
import json


class SimpleMediverseFormatter:
    """
    Simple, clean Mediverse-to-BioMistral formatter
    Task: Implement anatomy + timestamp encoder
    """


    def __init__(self):
        # ANATOMY ENCODER: Simple mapping dictionary (simulation names -> medical terms)
        self.anatomy_map = {
            # Foot parts
            "footskin": "foot skin",
            "metararsal1": "first metatarsal",
            "metatarsal1": "first metatarsal",
            "metatarsal1_remainingpiece": "remaining first metatarsal fragment",
            "proximalphalanx1": "proximal phalanx of first toe",
            # General foot areas
            "footr": "right foot",
            "footl": "left foot",
            "foot": "foot",
            # Non-anatomical / simulation objects
            "clinic_table": "surgical field",
            "unknown": "anatomical region"
        }
        # Known anatomical bases for fallback (substring -> term)
        self._anatomy_fallback_bases = [
            ("metatarsal", "metatarsal"),
            ("phalanx", "phalanx"),
            ("foot", "foot"),
            ("skin", "skin"),
        ]


        # Tool name cleanup
        self.tool_map = {
            "Scalpel": "scalpel",
            "BoneSaw": "bone saw",
            "OscillatingSaw": "oscillating saw",
            "Drill": "drill",
            "marker": "marking pen"
        }

        # Action normalization: Unity/simulation action names -> clinical action type
        self.action_map = {
            "Scalpel_incise": "incision",
            "BoneSaw_engage": "sawing",
            "ChevronCut_engage": "osteotomy",
            "Drill_insert": "drilling",
            # Legacy / alternate names (backward compatibility)
            "cutting_start": "cutting",
            "drilling_start": "drilling",
            "drawing_start": "drawing",
        }

    def _normalize_action(self, action):
        """Map raw action name to clinical action type for step description."""
        if not action:
            return ""
        for key, value in self.action_map.items():
            if key in action:
                return value
        return action.lower().replace("_", " ")

    def encode_anatomy(self, target):
        """
        ANATOMY ENCODER: Convert technical names to medical terms.
        Uses direct map, partial match, then fallback by known anatomical base.
        """
        if not target:
            return "anatomical region"
        target_lower = target.lower().strip()

        # Direct mapping
        if target_lower in self.anatomy_map:
            return self.anatomy_map[target_lower]

        # Partial matching for complex names (e.g. "clinic_table_2 (1)" -> clinic_table)
        for tech_name, medical_name in self.anatomy_map.items():
            if tech_name in target_lower:
                return medical_name

        # Fallback: if name contains known anatomical base, use that term
        for substr, term in self._anatomy_fallback_bases:
            if substr in target_lower:
                return term

        return "anatomical structure"


    def _normalize_time_fields(self, entry):
        """
        Normalize timestamps to always have start_time/end_time.
        """
        if 'start_time' in entry or 'end_time' in entry:
            start_time = entry.get('start_time', entry.get('timestamp'))
            end_time = entry.get('end_time', entry.get('timestamp'))
        else:
            start_time = entry.get('timestamp')
            end_time = entry.get('timestamp')

        normalized = dict(entry)
        normalized['start_time'] = start_time
        normalized['end_time'] = end_time

        if 'timestamp' not in normalized and start_time is not None:
            normalized['timestamp'] = start_time

        return normalized


    def encode_timestamps(self, json_data):
        """
        TIMESTAMP ENCODER: Sort by time and group similar actions
        """
        # Get data from JSON
        if 'data' in json_data:
            entries = json_data['data']
        elif 'Items' in json_data:
            entries = json_data['Items']
        else:
            entries = json_data


        normalized_entries = [self._normalize_time_fields(e) for e in entries]


        # Sort by start_time (fallback to timestamp)
        sorted_entries = sorted(normalized_entries, key=lambda x: float(x['start_time']))


        # Simple grouping: combine repeated actions on same target within 5 seconds
        grouped = []
        i = 0
        while i < len(sorted_entries):
            current = sorted_entries[i]
            similar_actions = [current]


            # Look for similar actions in next few entries
            j = i + 1
            while j < len(sorted_entries):
                next_entry = sorted_entries[j]
                time_diff = float(next_entry['start_time']) - float(current['start_time'])


                # Group if within 5 seconds and same action/target
                if (time_diff <= 5 and
                    next_entry['action'] == current['action'] and
                    next_entry.get('target', next_entry.get('bodypart')) == current.get('target', current.get('bodypart'))):
                    similar_actions.append(next_entry)
                    j += 1
                else:
                    break


            # Create grouped entry
            if len(similar_actions) > 1:
                group_end = similar_actions[-1]['end_time']
                grouped.append({
                    'start_time': current['start_time'],
                    'end_time': group_end,
                    'timestamp': current['start_time'],
                    'tool': current['tool'],
                    'action': current['action'],
                    'target': current.get('target', current.get('bodypart')),
                    'count': len(similar_actions),
                    'grouped': True
                })
            else:
                grouped.append(current)


            i = j if j > i + 1 else i + 1


        return grouped


    def convert_to_medical_step(self, entry):
        """
        Convert single entry to medical step description using normalized action type.
        """
        tool = entry['tool']
        action = entry['action']
        target = entry.get('target', entry.get('bodypart', 'unknown'))

        # Apply encoders
        medical_target = self.encode_anatomy(target)
        clean_tool = self.tool_map.get(tool, tool.lower())
        action_type = self._normalize_action(action)

        # Handle grouped actions (use normalized action type)
        if entry.get('grouped', False):
            count = entry.get('count', 1)
            if count > 1:
                if action_type == "incision" or action_type == "cutting":
                    return f"Multiple incisions made on {medical_target} using {clean_tool} ({count} cuts)"
                elif action_type == "drilling":
                    return f"Multiple drilling procedures on {medical_target} using {clean_tool} ({count} holes)"
                elif action_type == "sawing" or action_type == "osteotomy":
                    return f"Multiple osteotomy/sawing actions on {medical_target} using {clean_tool} ({count}x)"
                else:
                    return f"Multiple {action_type} actions on {medical_target} using {clean_tool} ({count}x)"

        # Single actions (branch on normalized action type; keep legacy *_start patterns)
        if action_type == "incision" or "cutting_start" in action or action_type == "cutting":
            return f"Incision made on {medical_target} using {clean_tool}"
        if action_type == "drilling" or "drilling_start" in action:
            return f"Drilling procedure performed on {medical_target} using {clean_tool}"
        if action_type == "drawing" or "drawing_start" in action:
            return f"Surgical site marked on {medical_target}"
        if action_type == "sawing":
            return f"Bone sawing performed on {medical_target} using {clean_tool}"
        if action_type == "osteotomy":
            return f"Chevron osteotomy performed on {medical_target} using {clean_tool}"
        return f"Surgical procedure on {medical_target} using {clean_tool}"


    def format_for_biomistral(self, json_data):
        """
        Main function: Convert JSON to BioMistral-ready prompt
        """
        # STEP 1: Timestamp encoding
        encoded_entries = self.encode_timestamps(json_data)


        # STEP 2: Convert to medical steps
        medical_steps = []
        for entry in encoded_entries:
            step = self.convert_to_medical_step(entry)
            medical_steps.append(step)


        # STEP 3: Create BioMistral prompt with numbered bullets
        steps_text = "\n".join([f"{i + 1}. {step}" for i, step in enumerate(medical_steps)])


        prompt = f"""You are a clinical documentation AI trained on surgical protocols and biomedical literature.


Task: Convert the following surgical steps into a brief, clinical narration suitable for a surgical operative note.


Instructions:
- Do NOT invent or infer any patient details, anatomical locations, or conditions.
- Do NOT describe any complications, findings, or outcomes unless specified.
- Use only the surgical actions provided below.
- Use precise medical terminology in a concise, professional tone.


Steps:
{steps_text}


Clinical Narration:"""


        return medical_steps, prompt

In [22]:
# import json

# class SimpleMediverseFormatter:
def test_formatter_from_file(json_filepath):
    """
    Test the formatter using data from a user-uploaded JSON file
    """
    formatter = SimpleMediverseFormatter()

    # Read JSON data from file
    with open(json_filepath, 'r') as infile:
        sample_data = json.load(infile)

    # Process with formatter
    steps, prompt = formatter.format_for_biomistral(sample_data)

    print("SIMPLE MEDIVERSE-TO-BIOMISTRAL FORMATTER TEST")

    print("\nFORMATTED MEDICAL STEPS:")
    for i, step in enumerate(steps, 1):
        print(f"{i}. {step}")

    print(f"\nBIOMISTRAL-READY PROMPT:")
    print(prompt)

    return steps, prompt

# Example usage:
if __name__ == "__main__":
    # filename
    test_formatter_from_file('surgery_events.json')

SIMPLE MEDIVERSE-TO-BIOMISTRAL FORMATTER TEST

FORMATTED MEDICAL STEPS:
1. Incision made on surgical field using scalpel
2. Incision made on foot skin using scalpel
3. Multiple osteotomy/sawing actions on first metatarsal using bone saw (2x)
4. Multiple osteotomy/sawing actions on first metatarsal using oscillating saw (2x)
5. Drilling procedure performed on first metatarsal using drill

BIOMISTRAL-READY PROMPT:
You are a clinical documentation AI trained on surgical protocols and biomedical literature.


Task: Convert the following surgical steps into a brief, clinical narration suitable for a surgical operative note.


Instructions:
- Do NOT invent or infer any patient details, anatomical locations, or conditions.
- Do NOT describe any complications, findings, or outcomes unless specified.
- Use only the surgical actions provided below.
- Use precise medical terminology in a concise, professional tone.


Steps:
1. Incision made on surgical field using scalpel
2. Incision made on foot

In [23]:
stepspip, promptpip = test_formatter_from_file('surgery_events.json')

SIMPLE MEDIVERSE-TO-BIOMISTRAL FORMATTER TEST

FORMATTED MEDICAL STEPS:
1. Incision made on surgical field using scalpel
2. Incision made on foot skin using scalpel
3. Multiple osteotomy/sawing actions on first metatarsal using bone saw (2x)
4. Multiple osteotomy/sawing actions on first metatarsal using oscillating saw (2x)
5. Drilling procedure performed on first metatarsal using drill

BIOMISTRAL-READY PROMPT:
You are a clinical documentation AI trained on surgical protocols and biomedical literature.


Task: Convert the following surgical steps into a brief, clinical narration suitable for a surgical operative note.


Instructions:
- Do NOT invent or infer any patient details, anatomical locations, or conditions.
- Do NOT describe any complications, findings, or outcomes unless specified.
- Use only the surgical actions provided below.
- Use precise medical terminology in a concise, professional tone.


Steps:
1. Incision made on surgical field using scalpel
2. Incision made on foot

In [24]:
promptpip

'You are a clinical documentation AI trained on surgical protocols and biomedical literature.\n\n\nTask: Convert the following surgical steps into a brief, clinical narration suitable for a surgical operative note.\n\n\nInstructions:\n- Do NOT invent or infer any patient details, anatomical locations, or conditions.\n- Do NOT describe any complications, findings, or outcomes unless specified.\n- Use only the surgical actions provided below.\n- Use precise medical terminology in a concise, professional tone.\n\n\nSteps:\n1. Incision made on surgical field using scalpel\n2. Incision made on foot skin using scalpel\n3. Multiple osteotomy/sawing actions on first metatarsal using bone saw (2x)\n4. Multiple osteotomy/sawing actions on first metatarsal using oscillating saw (2x)\n5. Drilling procedure performed on first metatarsal using drill\n\n\nClinical Narration:'

In [25]:
import os
os.environ["DISABLE_SAFETENSORS_CONVERSION"] = "1"

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "BioMistral/BioMistral-7B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    offload_folder="offload"
)

Loading weights: 100%|██████████| 291/291 [01:00<00:00,  4.83it/s, Materializing param=model.norm.weight]                              
Some parameters are on the meta device because they were offloaded to the disk.


In [26]:
prompt = promptpip
# Generate
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=3000,
    do_sample=True,
    top_p=0.9,
    temperature=0.7
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


You are a clinical documentation AI trained on surgical protocols and biomedical literature.


Task: Convert the following surgical steps into a brief, clinical narration suitable for a surgical operative note.


Instructions:
- Do NOT invent or infer any patient details, anatomical locations, or conditions.
- Do NOT describe any complications, findings, or outcomes unless specified.
- Use only the surgical actions provided below.
- Use precise medical terminology in a concise, professional tone.


Steps:
1. Incision made on surgical field using scalpel
2. Incision made on foot skin using scalpel
3. Multiple osteotomy/sawing actions on first metatarsal using bone saw (2x)
4. Multiple osteotomy/sawing actions on first metatarsal using oscillating saw (2x)
5. Drilling procedure performed on first metatarsal using drill


Clinical Narration: A longitudinal incision was made over the dorsal aspect of the first metatarsal. The foot was degloved to expose the metatarsal head. Two osteotomies

In [27]:
text = tokenizer.decode(outputs[0], skip_special_tokens=True)
prompt_out = text.split("Clinical Narration: ", 1)[1].strip()

In [28]:
prompt_out

'A longitudinal incision was made over the dorsal aspect of the first metatarsal. The foot was degloved to expose the metatarsal head. Two osteotomies were performed using a bone saw and an oscillating saw. A drilling procedure was then performed on the metatarsal head.'

In [29]:
import re
import spacy

nlp = spacy.load("en_core_web_sm")

NOMENCLATURE = {
    "drawing":       ("Marking",        ["Skin marking", "Demarcation"]),
    "incision":      ("Incision",       ["Cutaneous incision"]),
    "cutting":       ("Cutting",        ["Division"]),
    "sawing":        ("Osteotomy",      ["Sawing", "Bone cutting"]),
    "drilling":      ("Drilling",       ["Burring"]),
    "cauterization": ("Cauterization",  ["Coagulation", "Electrocautery"]),
    "irrigation":    ("Irrigation",     ["Lavage"]),
    "suction":       ("Suctioning",     ["Aspiration"]),
    "retraction":    ("Retraction",     ["Exposure"]),
    "closure":       ("Closure",        ["Wound closure", "Skin closure"]),
    "suturing":      ("Suturing",       ["Approximation"]),
    "dissection":    ("Dissection",     []),
    "removal":       ("Removal",        ["Extraction", "Explantation"]),
    "insertion":     ("Insertion",      ["Placement"]),
    "excision":      ("Excision",       ["Resection"]),
    "osteotomy":      ("Osteotomy",      ["Metatarsal osteotomy", "Chevron osteotomy", "Bone osteotomy"]),
}

VERBS = {
    "drawing":       ("Marked",       ["Outlined", "Demarcated"]),
    "incision":      ("Incised",      ["Opened"]),
    "cutting":       ("Cut",          ["Divided"]),
    "sawing":        ("Sawed",        ["Osteotomized", "Cut through"]),
    "drilling":      ("Drilled",      ["Burred"]),
    "cauterization": ("Cauterized",   ["Coagulated", "Electrocauterized"]),
    "irrigation":    ("Irrigated",    ["Lavaged"]),
    "suction":       ("Suctioned",    ["Aspirated"]),
    "retraction":    ("Retracted",    ["Exposed"]),
    "closure":       ("Closed",       ["Wound closed", "Skin closed"]),
    "suturing":      ("Sutured",      ["Approximated"]),
    "dissection":    ("Dissected",    []),
    "removal":       ("Removed",      ["Extracted", "Explant"]),
    "insertion":     ("Inserted",     ["Placed"]),
    "excision":      ("Excised",      ["Resected"]),
    "osteotomy":      ("Osteotomized",   ["Osteotomy", "Performed"]),
}

KNOWN_TOOLS = {"scalpel", "bone saw", "oscillating saw", "drill", "forceps", "retractor", "suction", "irrigator"}

def _is_summary_sentence(text: str, sentence_index: int = 0) -> bool:
    """Treat as summary/general if sentence is opener, repeated procedure, or wrap-up. Skip flagging for intro."""
    t = text.lower().strip()
    if not t:
        return False
    summary_phrases = [
        "the procedure was repeated",
        "the same steps",
        "the same technique",
        "repeated for the",
        "was repeated",
        "procedure was performed",
        "operative procedure was performed",
        "procedure was performed on the",
        "the following steps were performed",
        "as described above",
        "as previously",
        "following the same",
    ]
    if any(phrase in t for phrase in summary_phrases):
        return True
    # Allow first sentence as short introductory clause without tool/action
    if sentence_index == 1 and len(t) < 120:
        return True
    return False

def detect_actions(prompt_out: str):
    doc = nlp(prompt_out)
    sentences = list(doc.sents)
    results = []
    issues = []

    for i, sent in enumerate(sentences, 1):
        text = sent.text.strip()
        action_detected = None
        tool_detected = None
        for act in NOMENCLATURE.keys():
            if re.search(rf"\b{act}\w*\b", text, re.IGNORECASE):
                action_detected = act
                break
        for act, (verb, aliases) in VERBS.items():
            if any(re.search(rf"\b{alias.lower()}\b", text.lower()) for alias in [verb]+aliases):
                action_detected = act
                break

        for tool in KNOWN_TOOLS:
            if tool in text.lower():
                tool_detected = tool
                break

        is_summary = _is_summary_sentence(text, sentence_index=i)
        if not is_summary:
            if not action_detected:
                issues.append(f"Sentence {i}: Action not recognized -> '{text}'")
            if not tool_detected:
                issues.append(f"Sentence {i}: Tool not recognized -> '{text}'")

        results.append((i, text, action_detected, tool_detected))

    return results, issues

prompt_out = prompt_out
results, issues = detect_actions(prompt_out)

print("Detections:")
for r in results:
    print(r)

print("\nPotential Hallucinations:")
for i in issues:
    print(" -", i)

Detections:
(1, 'A longitudinal incision was made over the dorsal aspect of the first metatarsal.', 'incision', None)
(2, 'The foot was degloved to expose the metatarsal head.', None, None)
(3, 'Two osteotomies were performed using a bone saw and an oscillating saw.', 'osteotomy', 'bone saw')
(4, 'A drilling procedure was then performed on the metatarsal head.', 'osteotomy', 'drill')

Potential Hallucinations:
 - Sentence 2: Action not recognized -> 'The foot was degloved to expose the metatarsal head.'
 - Sentence 2: Tool not recognized -> 'The foot was degloved to expose the metatarsal head.'


In [30]:
import json
import os

def _get_entries_from_json(json_data):
    """Get list of entries from JSON (supports 'data' or 'Items' keys)."""
    if "data" in json_data:
        return json_data["data"]
    if "Items" in json_data:
        return json_data["Items"]
    return json_data

def _get_time_range(entry):
    if "start_time" in entry or "end_time" in entry:
        return entry.get("start_time", entry.get("timestamp")), entry.get("end_time", entry.get("timestamp"))
    t = entry.get("timestamp")
    return t, t

def create_enhanced_narration_from_formatter(json_filename, formatter=None, output_path=None):
    """
    Builds enhanced JSON using the same formatter (grouping + medical step strings)
    as the BioMistral prompt. Ensures alignment between prompt steps and timestamp sync.
    """
    if formatter is None:
        formatter = SimpleMediverseFormatter()
    try:
        with open(json_filename, "r") as f:
            json_data = json.load(f)
    except Exception as e:
        print("Error loading JSON.", e)
        return []
    encoded_entries = formatter.encode_timestamps(json_data)
    result = []
    for entry in encoded_entries:
        start_time = entry.get("start_time")
        end_time = entry.get("end_time")
        narration = formatter.convert_to_medical_step(entry)
        result.append({
            "start_time": start_time,
            "end_time": end_time,
            "timestamp": start_time,
            "narration": narration,
            "action": entry.get("action", ""),
            "tool": entry.get("tool", ""),
            "target": entry.get("target", entry.get("bodypart", "")),
        })
    base = os.path.splitext(os.path.basename(json_filename))[0]
    out_name = output_path or f"enhanced_{base}.json"
    with open(out_name, "w") as f:
        json.dump(result, f, indent=2)
    print(f"Saved to {out_name}")
    return result

def create_simple_narration_json(json_filename="surgery_events.json"):
    """
    Reads a JSON log and produces a list with start_time/end_time and narration.
    Narration is raw: "<action> on <target> using <tool>."
    For formatter-aligned output use create_enhanced_narration_from_formatter().
    """
    try:
        with open(json_filename, "r") as f:
            json_data = json.load(f)
    except Exception as e:
        print("Error loading JSON.", e)
        return []
    entries = _get_entries_from_json(json_data)
    try:
        sorted_entries = sorted(entries, key=lambda x: float(_get_time_range(x)[0]))
    except Exception as e:
        print("Error sorting timestamps.", e)
        return []

    result = []

    for entry in sorted_entries:
        start_time, end_time = _get_time_range(entry)
        action = entry.get("action", "")
        target = entry.get("target", entry.get("bodypart", ""))
        tool = entry.get("tool", "")

        narration = f"{action} on {target} using {tool}."

        result.append({
            "start_time": start_time,
            "end_time": end_time,
            "timestamp": start_time,
            "narration": narration,
            "action": action,
            "tool": tool,
            "target": target
        })

    base = os.path.splitext(json_filename)[0]
    out_name = f"enhanced_{base}.json"

    with open(out_name, "w") as f:
        json.dump(result, f, indent=2)

    print(f"Saved to {out_name}")
    return result

# Use formatter-aligned enhanced JSON so sync matches BioMistral prompt steps
pipeline = create_enhanced_narration_from_formatter("surgery_events.json")

Saved to enhanced_surgery_events.json


In [31]:
import json
import nltk
nltk.download('punkt_tab')
nltk.download("punkt")


def _get_time_range(entry):
    if "start_time" in entry or "end_time" in entry:
        start_time = entry.get("start_time", entry.get("timestamp"))
        end_time = entry.get("end_time", entry.get("timestamp"))
    else:
        start_time = entry.get("timestamp")
        end_time = entry.get("timestamp")
    return start_time, end_time


def group_events_by_action_tool(entries):
    """
    Merge consecutive events with the same (action, tool, target).
    Returns a list of merged entries with combined time ranges and narrations.
    """
    if not entries:
        return []
    sorted_entries = sorted(entries, key=lambda e: float(_get_time_range(e)[0]))
    grouped = []
    current_batch = [sorted_entries[0]]
    key = (
        current_batch[0].get("action", ""),
        current_batch[0].get("tool", ""),
        current_batch[0].get("target", current_batch[0].get("bodypart", "")),
    )
    for entry in sorted_entries[1:]:
        entry_key = (
            entry.get("action", ""),
            entry.get("tool", ""),
            entry.get("target", entry.get("bodypart", "")),
        )
        if entry_key == key:
            current_batch.append(entry)
        else:
            s0, e0 = _get_time_range(current_batch[0])
            s_last, e_last = _get_time_range(current_batch[-1])
            narrations = [e.get("narration", "") for e in current_batch]
            grouped.append({
                "start_time": s0,
                "end_time": e_last,
                "timestamp": s0,
                "narration": "; ".join(narrations),
                "action": key[0],
                "tool": key[1],
                "target": key[2],
            })
            current_batch = [entry]
            key = entry_key
    s0, e0 = _get_time_range(current_batch[0])
    s_last, e_last = _get_time_range(current_batch[-1])
    narrations = [e.get("narration", "") for e in current_batch]
    grouped.append({
        "start_time": s0,
        "end_time": e_last,
        "timestamp": s0,
        "narration": "; ".join(narrations),
        "action": key[0],
        "tool": key[1],
        "target": key[2],
    })
    return grouped


def merge_groups_to_target_count(entries, target_count):
    """
    If len(entries) > target_count, merge adjacent groups by smallest time gap
    until we have target_count groups.
    """
    if target_count >= len(entries) or target_count <= 0:
        return entries
    entries = list(entries)
    while len(entries) > target_count:
        gaps = []
        for i in range(len(entries) - 1):
            _, end_i = _get_time_range(entries[i])
            start_next, _ = _get_time_range(entries[i + 1])
            gaps.append((float(start_next) - float(end_i), i))
        gaps.sort(key=lambda x: x[0])
        merge_at = gaps[0][1]
        e1, e2 = entries[merge_at], entries[merge_at + 1]
        s1, e1_end = _get_time_range(e1)
        s2, e2_end = _get_time_range(e2)
        merged = {
            "start_time": s1,
            "end_time": e2_end,
            "timestamp": s1,
            "narration": e1.get("narration", "") + "; " + e2.get("narration", ""),
            "action": e1.get("action", ""),
            "tool": e1.get("tool", ""),
            "target": e1.get("target", ""),
        }
        entries = entries[:merge_at] + [merged] + entries[merge_at + 2:]
    return entries


def map_sentences_sequential(enhanced_json_path, narration_text, entries_override=None, output_path=None):
    """
    Maps each sentence of the large narration paragraph to timestamps.
    If entries_override is provided, use it instead of loading from file.
    When more events than sentences: merge groups to match sentence count.
    When more sentences than events: extra sentences get the last event's time range.
    output_path: path for final mapped JSON (default: final_mapped.json).
    """
    import warnings
    output_path = output_path or "final_mapped.json"
    if entries_override is not None:
        log_entries = entries_override
    else:
        with open(enhanced_json_path, "r") as f:
            log_entries = json.load(f)

    # Split narration text into sentences
    sentences = nltk.sent_tokenize(narration_text)
    sentences = [s.strip() for s in sentences if s.strip()]

    if len(log_entries) > len(sentences):
        log_entries = merge_groups_to_target_count(log_entries, len(sentences))
    if len(sentences) != len(log_entries):
        msg = f"Sentence count ({len(sentences)}) != event group count ({len(log_entries)}). Extra sentences use last group's time."
        warnings.warn(msg, UserWarning)
        print(f"[WARNING] {msg}")

    final_output = []

    for i, sentence in enumerate(sentences):
        if i < len(log_entries):
            start_time, end_time = _get_time_range(log_entries[i])
        else:
            start_time, end_time = _get_time_range(log_entries[-1])

        final_output.append({
            "start_time": start_time,
            "end_time": end_time,
            "timestamp": start_time,
            "sentence": sentence
        })

    # Save final JSON
    with open(output_path, "w") as f:
        json.dump(final_output, f, indent=2)

    print(f"[2] Final mapped JSON saved as: {output_path}")
    return final_output

# If we want overlay from final_mapped.json later,

# Well need to change one line in overlay:
# 	•	From narration = entry.get("narration")
# 	•	To text = entry.get("sentence") 

def generate_overlay_timeline(entries, output_path="overlay_timeline.json"):
    """
    Generates a time-based overlay from narration entries.
    A narration is active if start_time <= t < end_time.
    The overlay changes only at boundary times.
    """

    valid_entries = []
    for entry in entries:
        start_time = entry.get("start_time")
        end_time = entry.get("end_time")
        # narration = entry.get("narration")
        narration = entry.get("sentence")
        if start_time is None or end_time is None or narration is None:
            continue
        valid_entries.append({
            "start_time": float(start_time),
            "end_time": float(end_time),
            "narration": narration
        })

    if not valid_entries:
        with open(output_path, "w") as f:
            json.dump([], f, indent=2)
        print(f"Overlay timeline saved as: {output_path}")
        return []

    boundary_times = sorted({
        t
        for e in valid_entries
        for t in (e["start_time"], e["end_time"])
    })

    timeline = []
    for t in boundary_times:
        active = [
            e["narration"]
            for e in valid_entries
            if e["start_time"] <= t < e["end_time"]
        ]
        timeline.append({"time": t, "content": active})

    with open(output_path, "w") as f:
        json.dump(timeline, f, indent=2)

    print(f"Overlay timeline saved as: {output_path}")
    return timeline

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/swethashankar/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/swethashankar/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [32]:
with open("enhanced_surgery_events.json", "r") as f:
    enhanced_entries = json.load(f)

grouped = group_events_by_action_tool(enhanced_entries)
map_sentences_sequential("enhanced_surgery_events.json", prompt_out, entries_override=grouped)

with open("final_mapped.json", "r") as f:
    mapped_entries = json.load(f)

generate_overlay_timeline(mapped_entries, "overlay_timeline.json")

[2] Final mapped JSON saved as: final_mapped.json
Overlay timeline saved as: overlay_timeline.json


[{'time': 0.0,
  'content': ['A longitudinal incision was made over the dorsal aspect of the first metatarsal.']},
 {'time': 14.279999732971191, 'content': []},
 {'time': 19.920000076293945,
  'content': ['The foot was degloved to expose the metatarsal head.']},
 {'time': 20.53999900817871, 'content': []},
 {'time': 27.19999885559082,
  'content': ['Two osteotomies were performed using a bone saw and an oscillating saw.']},
 {'time': 29.85999870300293, 'content': []},
 {'time': 35.21999740600586,
  'content': ['A drilling procedure was then performed on the metatarsal head.']},
 {'time': 37.34000015258789, 'content': []}]

In [33]:
# Pipeline config: paths and generation params (override as needed)
PIPELINE_CONFIG = {
    "json_path": "surgery_events.json",
    "enhanced_path": "enhanced_surgery_events.json",
    "final_mapped_path": "final_mapped.json",
    "overlay_path": "overlay_timeline.json",
    "model_name": "BioMistral/BioMistral-7B",
    "max_new_tokens": 6000,
    "top_p": 0.9,
    "temperature": 0.7,
}

def run_narration_pipeline(json_path=None, config=None, tokenizer=None, model=None):
    """
    Single entry point: load JSON -> formatter -> BioMistral -> extract narration
    -> hallucination check -> enhanced JSON (formatter) -> map sentences -> overlay.
    Uses global tokenizer/model if not provided. config overrides PIPELINE_CONFIG.
    """
    cfg = {**PIPELINE_CONFIG, **(config or {})}
    json_path = json_path or cfg["json_path"]
    tok = tokenizer if tokenizer is not None else globals().get("tokenizer")
    mod = model if model is not None else globals().get("model")
    if tok is None or mod is None:
        raise RuntimeError("Tokenizer and model must be loaded (or passed in) before run_narration_pipeline.")

    with open(json_path, "r") as f:
        json_data = json.load(f)
    formatter = SimpleMediverseFormatter()
    medical_steps, prompt = formatter.format_for_biomistral(json_data)

    inputs = tok(prompt, return_tensors="pt").to(mod.device)
    outputs = mod.generate(
        **inputs,
        max_new_tokens=cfg["max_new_tokens"],
        do_sample=True,
        top_p=cfg["top_p"],
        temperature=cfg["temperature"],
    )
    text = tok.decode(outputs[0], skip_special_tokens=True)
    narration = text.split("Clinical Narration:", 1)[1].strip() if "Clinical Narration:" in text else text

    results, issues = detect_actions(narration)
    if issues:
        print("[Hallucination check] Potential issues:", issues)

    enhanced_entries = create_enhanced_narration_from_formatter(
        json_path, formatter=formatter, output_path=cfg["enhanced_path"]
    )
    grouped = group_events_by_action_tool(enhanced_entries)
    mapped = map_sentences_sequential(
        cfg["enhanced_path"], narration, entries_override=grouped, output_path=cfg["final_mapped_path"]
    )
    with open(cfg["final_mapped_path"], "r") as f:
        mapped_entries = json.load(f)
    timeline = generate_overlay_timeline(mapped_entries, cfg["overlay_path"])
    return {"narration": narration, "mapped": mapped, "timeline": timeline, "issues": issues}

## Unity VR output contract

Use these schemas when consuming narration from the pipeline in Mediverse VR.

### final_mapped.json
One object per sentence, in chronological order. Each entry:
- `start_time` (float): Start time in seconds for this sentence.
- `end_time` (float): End time in seconds.
- `timestamp` (float): Same as `start_time`.
- `sentence` (string): One clinical narration sentence to show for this time range.

Unity can look up the current playback time and display the `sentence` whose `start_time` <= t < `end_time`.

### overlay_timeline.json
Time-ordered list of boundaries where the visible narration changes. Each entry:
- `time` (float): Boundary time in seconds.
- `content` (array of strings): Narration text(s) active at this time (usually one). Empty array means no narration at that instant.

To drive a UI: at playback time `t`, find the largest `time` <= t and use that entry’s `content`; when `t` passes the next `time`, update to the next entry’s `content`.

In [34]:
# Local TTS: writes `audio_steps/*.wav` and sidecar `audio_manifest.json`
# Prereq (once): `pip install -r requirements-tts.txt`
import subprocess
import sys
from pathlib import Path

cmd = [
    sys.executable,
    str(Path("generate_step_audio.py").resolve()),
    "--input",
    "final_mapped.json",
    "--output-manifest",
    "audio_manifest.json",
    "--audio-dir",
    "audio_steps",
]
subprocess.run(cmd, check=True)

/Users/swethashankar/Documents/OPT/MediverseVR/.venv/lib/python3.12/site-packages/numpy/lib/_format_impl.py:838: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  array = pickle.load(fp, **pickle_kwargs)
ə lɑnd͡ʒətudənəl ɪnsɪʒən wəz meɪd oʊvɚ ðə dɔɹsəl æspɛkt əv ðə fɚst mɛtətɚzəl.
Character '͡' not found in the vocabulary. Discarding it.


Wrote manifest: audio_manifest.json (4 narration clip(s) == 4 WAV file(s))
Audio directory: audio_steps


CompletedProcess(args=['/Users/swethashankar/Documents/OPT/MediverseVR/.venv/bin/python', '/Users/swethashankar/Documents/OPT/MediverseVR/generate_step_audio.py', '--input', 'final_mapped.json', '--output-manifest', 'audio_manifest.json', '--audio-dir', 'audio_steps'], returncode=0)

In [35]:
# ── Cleanup stale WAVs + verify text/audio match ────────────────────────────
import json
import os
from pathlib import Path


def cleanup_stale_audio(
    manifest_path="audio_manifest.json",
    audio_dir="audio_steps",
    dry_run=False,
):
    """
    Removes WAV files in audio_dir that are NOT referenced by audio_manifest.json.
    Set dry_run=True to preview what would be deleted without actually deleting.
    """
    with open(manifest_path, "r") as f:
        manifest = json.load(f)

    # Collect filenames that the current manifest references
    referenced = set()
    for step in manifest.get("steps", []):
        ap = step.get("audio_path", "")
        if ap:
            referenced.add(Path(ap).name)

    audio_path = Path(audio_dir)
    all_wavs = list(audio_path.glob("*.wav")) if audio_path.exists() else []
    stale = [w for w in all_wavs if w.name not in referenced]

    if not stale:
        print(f"[cleanup] No stale WAV files found in {audio_dir}/")
        return

    action = "Would remove" if dry_run else "Removing"
    print(f"[cleanup] {action} {len(stale)} stale WAV file(s):")
    for w in stale:
        print(f"  - {w.name}")
        if not dry_run:
            w.unlink()
    if not dry_run:
        print(f"[cleanup] Done. {len(stale)} file(s) removed.")


def verify_text_audio_match(
    mapped_path="final_mapped.json",
    manifest_path="audio_manifest.json",
    audio_dir="audio_steps",
):
    """
    Checks that:
      1. final_mapped.json sentence count == audio_manifest.json clip count
      2. Every WAV file referenced in the manifest actually exists on disk
    Warns (but does not fail) if extra unreferenced WAV files are found.
    Raises AssertionError on any real mismatch.
    """
    # 1. Count non-empty sentences in final_mapped.json
    with open(mapped_path, "r") as f:
        mapped = json.load(f)
    text_count = sum(
        1 for entry in mapped
        if (entry.get("sentence") or entry.get("narration") or "").strip()
    )

    # 2. Collect successful steps from the manifest
    with open(manifest_path, "r") as f:
        manifest = json.load(f)
    manifest_clips = manifest.get("expected_audio_clips", 0)
    successful_steps = [
        s for s in manifest.get("steps", [])
        if s.get("audio_path") and not s.get("error")
    ]
    manifest_audio_count = len(successful_steps)

    # 3. Check that each manifest-referenced WAV actually exists on disk
    base = Path(audio_dir)
    missing_on_disk = []
    for step in successful_steps:
        wav = Path(step["audio_path"])
        if not wav.is_absolute():
            wav = base / wav.name
        if not wav.exists():
            missing_on_disk.append(wav.name)

    # 4. Warn about extra (unreferenced) WAV files on disk
    referenced_names = {Path(s["audio_path"]).name for s in successful_steps}
    all_wavs = sorted(base.glob("*.wav")) if base.exists() else []
    extra_wavs = [w.name for w in all_wavs if w.name not in referenced_names]

    print(f"Text sentences  in {mapped_path}:      {text_count}")
    print(f"Manifest clips  in {manifest_path}: {manifest_audio_count} (expected={manifest_clips})")
    print(f"Manifest WAVs missing on disk:             {len(missing_on_disk)}")
    print(f"Extra unreferenced WAVs in {audio_dir}/:  {len(extra_wavs)}"
          + (f" <- run cleanup_stale_audio() to remove" if extra_wavs else ""))

    issues = []
    if text_count == 0:
        issues.append(
            f"  - {mapped_path} is empty! Run the full pipeline "
            f"(BioMistral -> map_sentences_sequential) first."
        )
    if manifest_audio_count != text_count:
        issues.append(
            f"  - Manifest clip count ({manifest_audio_count}) != "
            f"text sentence count ({text_count}). Re-run generate_step_audio.py."
        )
    if missing_on_disk:
        issues.append(
            f"  - {len(missing_on_disk)} WAV file(s) listed in manifest are missing "
            f"from disk: {missing_on_disk}"
        )

    if issues:
        print("\n[MISMATCH DETECTED]")
        for issue in issues:
            print(issue)
        raise AssertionError(
            f"Text/audio mismatch: {text_count} sentence(s), "
            f"{manifest_audio_count} manifest clip(s), "
            f"{len(missing_on_disk)} missing WAV(s) on disk."
        )

    print(f"\n[OK] {text_count} sentence(s) matched by {manifest_audio_count} audio clip(s). All consistent.")
    for i, step in enumerate(successful_steps):
        t = step["text"]
        preview = (t[:80] + "...") if len(t) > 80 else t
        print(f"  {i+1}. {Path(step['audio_path']).name}  ->  \"{preview}\"")


# ── Run cleanup then verify ───────────────────────────────────────────────────
cleanup_stale_audio(
    manifest_path="audio_manifest.json",
    audio_dir="audio_steps",
    dry_run=False,   # set True to preview without deleting
)

verify_text_audio_match(
    mapped_path="final_mapped.json",
    manifest_path="audio_manifest.json",
    audio_dir="audio_steps",
)


[cleanup] Removing 6 stale WAV file(s):
  - step_0004_19b4465305485565.wav
  - step_0003_c3716ab1bd349410.wav
  - step_0002_32dd6f0ec85c80d3.wav
  - step_0001_28796554fda596d8.wav
  - step_0000_4b569b874b785866.wav
  - step_0005_e4e3e7a3084da139.wav
[cleanup] Done. 6 file(s) removed.
Text sentences  in final_mapped.json:      4
Manifest clips  in audio_manifest.json: 4 (expected=4)
Manifest WAVs missing on disk:             0
Extra unreferenced WAVs in audio_steps/:  0

[OK] 4 sentence(s) matched by 4 audio clip(s). All consistent.
  1. step_0000_462d6790bd949b90.wav  ->  "A longitudinal incision was made over the dorsal aspect of the first metatarsal."
  2. step_0001_17df2df9fd092b5c.wav  ->  "The foot was degloved to expose the metatarsal head."
  3. step_0002_99b29571f2e34467.wav  ->  "Two osteotomies were performed using a bone saw and an oscillating saw."
  4. step_0003_acfd331ca23074de.wav  ->  "A drilling procedure was then performed on the metatarsal head."
